# Quantum Oracle Sketching for Boolean Functions

> **Quantum Oracle Sketching** is a data-loading algorithm introduced by Zhao et al. [[1]](#ref1). It provides the missing link between classical data access and the *coherent oracle queries* on which many powerful quantum algorithms are built.
>
> The algorithm approximates the desired oracle $O_f$ on the fly from a stream of classical samples $(x_t, f(x_t))$, applying an incremental data-dependent rotation per sample and discarding each sample once consumed, at no point is the dataset stored in either classical or quantum memory. The result is an exponential advantage in space complexity over any classical learner, and when the data distribution varies in time while the learner stays fixed obtains a super-polynomial advantage in sample complexity as well: a classical machine requires the number of data examples to scale super-polynomially with the data size $N$, while a quantum learner only needs an amount of data that scales linearly, achieving matching accuracy and probabilistic performance.
>
> The algorithm treats the following problem:
>
> - **Input:** $M$ classical data samples $\{(x_t, f(x_t))\}_{t=1}^{M}$, where each $x_t \in [N] = \{0,1,\dots,N-1\}$ is drawn independently from a (possibly time-varying) distribution $p$, and $f$ is the target function. Here we consider a Boolean ($f:[N]\to\{0,1\}$) or real-valued ($f:[N]\to\mathbb{R}$ target function. There are extensions for a real vector $\vec f \in \mathbb{R}^N$), or matrix-valued ($f:[N]\times[N]\to\mathbb{R}$, equivalently a real matrix in $\mathbb{R}^{N\times N}$).
> - **Output:** A quantum unitary $V$ on $n = \lceil\log_2 N\rceil$ qubits that approximates the corresponding *coherent* query oracle $O$, the phase oracle $|x\rangle \to (-1)^{f(x)}|x\rangle$, or a real valued phase $|x\rangle \to e^{i \theta_x}|x\rangle$ to error $\epsilon$ in diamond distance. In the extenstions to vectors and martrices the oracle can output a state-preparation unitary $|0\rangle \to \sum_x f(x)|x\rangle/\|f\|$, or a block encoding of the matrix. $V$ can then be used as a drop-in subroutine inside any quantum query algorithm.
>
> **Complexity.**
>
> - **Sample complexity**: A single $\epsilon$-accurate oracle sketch requires $M = \Theta(N/\epsilon)$ classical samples. To support an arbitrary quantum algorithm that makes $Q$ queries to total error $\epsilon$, each query is sketched to error $\epsilon/Q$, giving
> $$ M_\text{total} = \Theta\!\bigl(N Q^2 / \epsilon\bigr). $$
> The quadratic dependence on $Q$ is unavoidable, mirroring the Born-rule relationship between quantum amplitudes and probabilities.
> - **Space complexity**: $\mathcal{O}(\mathrm{poly}\log N)$ qubits — the quantum machine never stores the dataset, only its current quantum state.
> - **Time complexity**: $\widetilde{\mathcal{O}}(N)$ in the data-loading stage (one constant-depth controlled gate per sample); subsequent processing of each sample requires only $\mathrm{poly}\log N$ time. Here the overscript tilde designates possible hidden logarithmic dependencies in $N$.
> - **Classical lower bound (dynamic case)**: Any classical machine of size ${\cal{O}}(N^{0.99})$ that matches the quantum prediction accuracy on a time-varying distribution, requires a super-polynomial number of samples.
>
> ---
>
> **Keywords:** Quantum Learning Theory, Quantum Machine Learning (QML), Streaming Algorithms

## Introduction

Most quantum speedup over a classical baseline is stated relative to an *oracle*, a unitary $O_f$ that encodes the input function $f$ and acts in superposition. [Grover's search](https://github.com/Classiq/classiq-library/blob/main/algorithms/search_and_optimization/grover/grover.ipynb) assumes a phase oracle $O_f|x\rangle = (-1)^{f(x)}|x\rangle$; [Shor's period-finding](https://github.com/Classiq/classiq-library/blob/main/algorithms/number_theory_and_cryptography/shor/shor.ipynb) queries a modular-exponentiation oracle; the [HHL linear-system solver](https://github.com/Classiq/classiq-library/blob/main/algorithms/quantum_linear_solvers/hhl/hhl.ipynb) [[5]](#ref5), [quantum singular-value transformation (QSVT)](https://github.com/Classiq/classiq-library/blob/main/algorithms/quantum_linear_solvers/qsvt_matrix_inversion/qsvt_matrix_inversion.ipynb) [[4]](#ref4), and most quantum machine-learning routines rely on a block-encoding oracle for the input matrix; [amplitude estimation](https://github.com/Classiq/classiq-library/blob/main/algorithms/amplitude_amplification_and_estimation/oblivious_amplitude_amplification/oblivious_amplitude_amplification.ipynb) and quantum walks similarly assume coherent access to the input. This *coherent* access, the ability to evaluate $f$ on a superposition of inputs in a single query, is exactly what enables the quadratic and exponential speedups, without it, an in depth analysis shows that the algorithms collapse to their classical counterparts.

Yet real-world classical data, produced one sample at a time by experiments, sensors, or users, does not natively support such queries. The standard remedy is to load the entire dataset into a quantum random-access memory (QRAM [[3]](#ref3)), but its fault-tolerance overhead often exceeds the quantum advantage it enables, leaving the most powerful quantum algorithms without a practical interface to massive classical data.

Quantum oracle sketching is the algorithm that closes this gap: it constructs an approximate $O_f$ on the fly from a stream of classical samples, with no QRAM and no full dataset ever held in memory.

## The streaming-learning framework

Quantum oracle sketching is proposed in the context of  a specific online-learning model. The framework sets the rules under which both the quantum and the classical baselines operate, and it explains why the comparison between the two is well-posed.

**Setup.**

- A *data-generating process* $\mathcal{D}$ emits a sequence of classical samples $z_1, z_2, \dots, z_M$. Each sample is one observation of the underlying object — a function value $z_t = (x_t, f(x_t))$ for a Boolean target, a feature-vector / label pair $z_t = (i_t, \vec{x}_{i_t}, y_{i_t})$ for classification, or an entry $z_t = (i_t, j_t, A_{i_t j_t}, k_t, b_{k_t})$ of a linear system $A\vec{x} = \vec{b}$.
- A *learner* of size $S$ (which can be the number of logical qubits composing the quantum learner, or the number of floating-point words for a classical one) observes the samples *one at a time*. After seeing $z_t$ it updates its memory, then discards $z_t$. At no point is the dataset stored in full. This is known as the *streaming* model.
- After $M$ samples, the learner is asked to predict, classify, or compress some property of the underlying process $\mathcal{D}$ (not of the empirical sample!). Examples: the label of a held-out test feature vector (classification), the value of a quadratic form $\vec{x}^\top \mathcal{M} \vec{x}$ (linear systems), or the projection of a test point onto the top principal direction (PCA).

The two resources of interest are the *machine size* $S$ and the *sample count* $M$. Optimally, we would like to minimize both simultaneously.

**Static versus dynamic data.** The framework allows the data distribution to drift in time, characterized by two parameters:

- *Refreshing time* $\tau$: the timescale beyond which samples become effectively uncorrelated.
- *Repetition number* $R$: the maximum expected number of times any single sample is repeated within a window of $\tau$ steps.

When $\tau \to \infty$ we recover the static, i.i.d. setting. When $\tau = \widetilde{\mathcal{O}}(N)$ the distribution refreshes on the same timescale as the data size, and the quantum advantage in sample complexity becomes *super-polynomial*. In this regime, any $\Omega(N^{0.99})$-size classical machine needs super-polynomially many samples while a quantum learner with $\mathrm{poly}\log N$ qubits suffice with $\widetilde{\mathcal{O}}(N)$ samples.

**End-to-end pipeline.** The full quantum learner is a three-stage assembly:

1. **Quantum oracle sketching** consumes the stream of classical samples and produces an approximate oracle unitary $V \approx O_f$ (the subject of the present notebook).
2. A **quantum query algorithm** — Grover, QSVT [[4]](#ref4), HHL [[5]](#ref5), amplitude estimation, etc. — uses $V$ as a black-box subroutine to prepare a target state $|\psi_\text{target}\rangle$ that encodes the desired property of $\mathcal{D}$.
3. **Interferometric classical shadows** a sign-preserving variant of the Classical Shadow Tomography protocol [[2]](#ref2)). The method extracts a compact classical model from $|\psi_\text{target}\rangle$ in $\mathrm{poly}\log N$ measurements, ready for offline prediction on arbitrary test inputs.

This notebook covers a part of the first stage in detail: the Boolean and real-valued phase oracles in full.

## Implementation of the Oracle Sketching Algorithm

### Oracle Sketching of a Boolean Function and Uniform Distribution

We aim to implement the Boolean phase oracle

$$
O\,|x\rangle = (-1)^{f(x)}\,|x\rangle, \qquad x \in [N],\ f : [N] \to \{0,1\},
$$

given only random classical samples $z_t = (x_t,\, f(x_t))$ for $t = 1,\dots,M$, where each $x_t$ is drawn uniformly from $[N] = \{0,1,\dots,N-1\}$.

**Per-sample rotation.** For each sample we apply a small data-dependent phase rotation that fires only on the basis state $|x_t\rangle$:

$$
V_t \;=\; \exp\!\bigl(i\,\tau\, f(x_t)\,|x_t\rangle\!\langle x_t| / M\bigr). \tag{1}
$$

**Coherent accumulation.** Because every $V_t$ is diagonal in the computational basis, all factors commute and the product collapses cleanly:

$$
V \;\equiv\; \prod_{t=1}^M V_t \;=\; \exp\!\Bigl(i\tfrac{\tau}{M}\sum_{t=1}^M f(x_t)\,|x_t\rangle\!\langle x_t|\Bigr) \;=\; \sum_{x=0}^{N-1} \exp\bigl(i\,\tau\, m_x\, f(x)\bigr)\,|x\rangle\!\langle x|, \tag{2}
$$

where $m_x = \frac{1}{M}\sum_t \mathbf{1}[x_t = x]$ is the empirical frequency of basis label $x$.

**Concentration.** As $M$ grows, $m_x \to p(x) = 1/N$ by the law of large numbers. Picking $\tau = \pi N$ makes $\tau\, m_x \to \pi$ and

$$
V \;\longrightarrow\; \sum_{x=0}^{N-1} e^{i\pi f(x)}\,|x\rangle\!\langle x| \;=\; \sum_x (-1)^{f(x)}\,|x\rangle\!\langle x| \;=\; O. \tag{3}
$$

**Sample complexity.** Ref. [[1]](#ref1) shows the diamond-distance error decays as $\epsilon = \mathcal{O}(N/M)$, so reaching error $\epsilon$ costs

$$
M = \Theta(N/\epsilon) \tag{4}
$$

samples. Each sample is processed once and immediately discarded.

**Comparison to generic random Hamiltonian simulation.** The $N/M$ scaling above is *not* what a naive random-rotation strategy would deliver. If one instead applied a sequence of small phases $\exp(i\,\tau\, h_{z_t}/M)$ driven by an arbitrary Hamiltonian $h_z$ with $\|h_z\| = \mathcal{O}(1)$ — hoping the sequence approximates $\exp(i\,\tau\,\mathbb{E}[h_z])$, randomized-Hamiltonian-simulation results show the operator-norm error compounds to

$$
\epsilon_\text{naive} \;=\; \mathcal{O}(N^2 / M),\qquad \text{i.e.}\qquad M_\text{naive} \;=\; \Theta(N^2 / \epsilon).
$$

That extra factor of $N$ would consume the entire quantum advantage. To support $Q$ queries with combined error $\epsilon$, each individual query must be sketched to error $\epsilon/Q$, so per-query naive simulation needs $\Theta(N^2 Q / \epsilon)$ samples and the total becomes

$$
M_\text{total}^\text{naive} \;=\; Q \cdot \Theta(N^2 Q/\epsilon) \;=\; \Theta(N^2 Q^2/\epsilon),
$$

versus the $\Theta(NQ^2/\epsilon)$ achieved here — a factor-$N$ blow-up that would take $\widetilde{\mathcal{O}}(N)$-sample tasks straight back into the regime where any classical machine of size $\Omega(N^{0.99})$ wins. The improvement to $\epsilon = \mathcal{O}(N/M)$ comes from a crucial piece of structure: each $V_t$ phases only the *one-dimensional* subspace $|x_t\rangle\!\langle x_t|$, and distinct values of $x$ label *mutually orthogonal* subspaces, so per-sample errors live in disjoint Hilbert-space blocks rather than compounding across the full register. Quantum oracle sketching is, in this sense, a carefully engineered *non-generic* instance of randomized Hamiltonian simulation, designed precisely to dodge the $N^2/M$ trap.

We begin by importing the required python modules and define the size of the problem $N$ and the truth table, defining $f$.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from classiq import *

np.random.seed(7)

In [ ]:
N = 8  # problem size; the register has int(log2(N)) qubits

# A random Boolean target function f: [N] -> {0, 1}
F_TABLE = np.random.randint(0, 2, size=N)
print(f"f(x) for x = 0,...,{N-1}:  {F_TABLE.tolist()}")

We begin by introducing the utility functions. The primitive `apply_basis_phase` realises $\exp(i\theta\,|x_\text{int}\rangle\!\langle x_\text{int}|)$ by controlling on the equality predicate `qvar == x_int`; the body `phase(theta)` becomes a relative phase on the controlled subspace and the identity elsewhere, which is exactly $V_t$. 
The `quantum_oracle_sketch_boolean` iterates over the samples of `x_samples`, leading to an approximate application of $O_f$.

In [3]:
@qfunc
def apply_basis_phase(theta: CReal, x_int: CInt, qvar: QNum) -> None:
    """Apply ``exp(i · θ · |x_int⟩⟨x_int|)`` on ``qvar``.

    Implemented by controlling on the equality predicate ``qvar == x_int``;
    the inner ``phase(θ)`` becomes a relative phase on the controlled
    subspace and the identity elsewhere, which is exactly ``V_t``.
    """
    control(qvar == x_int, lambda: phase(theta))


@qfunc
def quantum_oracle_sketch_boolean(
    theta: CReal, x_samples: CArray[CInt], qvar: QNum
) -> None:
    """Sketched Boolean phase oracle ``V = ∏_t exp(i θ |x_t⟩⟨x_t|)``.

    ``x_samples`` should be pre-filtered to the indices with ``f(x_t) = 1``;
    a single fixed angle ``θ = π N / M`` is then applied per sample.
    """
    repeat(
        count=x_samples.len,
        iteration=lambda t: apply_basis_phase(theta, x_samples[t], qvar),
    )

We pre-filter the classical data so only samples with $f(x_t)=1$ enter the circuit (the rest contribute the identity), fix $\theta = \pi N / M$, and bake the samples into the circuit at synthesis time. The leading Hadamard transform lets us see the diagonal-phase action of $V$ on the uniform superposition $|+\rangle^{\otimes n}$.

In [ ]:
# For circuit synthesis we use a smaller M so the visualization stays readable;
# the numpy benchmark below validates the algorithm at larger M.
M_demo = 200
demo_samples = np.random.randint(0, N, size=M_demo)
positive_samples = demo_samples[F_TABLE[demo_samples] == 1].tolist()
theta = np.pi * N / M_demo  # τ / M with τ = π N
n_qubits = int(np.log2(N))


@qfunc
def main(qvar: Output[QNum]) -> None:
    allocate(n_qubits, qvar)
    hadamard_transform(qvar)
    quantum_oracle_sketch_boolean(theta, positive_samples, qvar)


qprog = synthesize(main)
show(qprog)

**Verification on a 2-qubit Bell state via Classiq execution.**

Take $f = [0, 0, 1, 1]$ — a phase on the most-significant-bit subspace. The ideal oracle maps $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ to $|\Phi^-\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$. We prepare $|\Phi^+\rangle$ on a 2-qubit register, apply the sketched `quantum_oracle_sketch_boolean` for one random sample sequence of $M = 500$ shots, then run the *inverse* Bell preparation (CX followed by H on qubit 0) so that the Bell basis is rotated back to the computational basis: in particular $|\Phi^-\rangle \mapsto |10\rangle$ (i.e. `qvar = 1`). Measuring the register in the computational basis with the new top-level `sample(...)` helper then makes the fidelity $|\langle\Phi^-|V|\Phi^+\rangle|^2$ equal to the probability of observing `qvar = 1`. The single-realization error scales as $\sqrt{N/M}$, so the fidelity gap from $1$ is of order $N/M \approx 0.008$.

In [ ]:
from classiq.execution import sample

# 2-qubit Bell-state setup: f = [0,0,1,1] phases the MSB subspace.
N_ex, M_ex = 4, 500
F_ex = np.array([0, 0, 1, 1])
theta_ex = np.pi * N_ex / M_ex

np.random.seed(0)
demo_samples_ex = np.random.randint(0, N_ex, size=M_ex)
positive_samples_ex = demo_samples_ex[F_ex[demo_samples_ex] == 1].tolist()


@qfunc
def main(qvar: Output[QNum]) -> None:
    bell = QArray[QBit, 2]()
    allocate(2, bell)
    # 1) Prepare |Φ+⟩.
    H(bell[0])
    CX(bell[0], bell[1])
    # 2) Apply the sketched oracle on the QNum view.
    bind(bell, qvar)
    quantum_oracle_sketch_boolean(theta_ex, positive_samples_ex, qvar)
    # 3) Inverse Bell prep — rotates the Bell basis to the computational basis,
    #    in particular |Φ-⟩ → |10⟩ (qvar = 1).
    bind(qvar, bell)
    CX(bell[0], bell[1])
    H(bell[0])
    bind(bell, qvar)


qprog_bell = synthesize(main)

# New top-level execute helper: returns a counts dataframe.
df = sample(qprog_bell, "simulator", num_shots=10_000)

fidelity = float(df.loc[df["qvar"] == 1, "probability"].sum())
print(f"Fidelity |⟨Φ-|V|Φ+⟩|² at M={M_ex}:  {fidelity:.4f}")

### Numpy reference and sample-complexity scaling

Before launching the Classiq circuit, we build the sketched unitary $V$ classically by collecting $M$ random samples and assembling the diagonal matrix from eq. (2). This is the ground truth that the quantum circuit reproduces.

Two scalings of the error to the ideal oracle $O$ are worth keeping distinct:

- **Single-realization** operator-norm error $\|V_\lambda - O\|_2$ for one random data sample $\lambda = (x_1, \dots, x_M)$ is dominated by the *variance* of the empirical frequencies $|m_x - 1/N| = \mathcal{O}(1/\sqrt{NM})$. Each diagonal entry deviates by $\pi N|m_x - 1/N| \sim \sqrt{N/M}$, so the spectral norm scales as $\|V_\lambda - O\|_2 = \mathcal{O}(\sqrt{N/M})$ (up to a $\sqrt{\log N}$ factor from the max over $\sim N$ bins).

- **Random-unitary channel** $\mathcal{C}(\rho) = \mathbb{E}_\lambda[V_\lambda\, \rho\, V_\lambda^\dagger]$ — the actual object that gets used inside any quantum query algorithm — has diamond-distance error $\mathcal{O}(N/M)$ to $O \rho O^\dagger$. The improvement over a single realization comes because the random unitary is *unbiased to leading order*: $\|\mathbb{E}[V_\lambda] - O\| = \mathcal{O}(N/M)$, an order of magnitude better than a single draw.

This second scaling drives the $M = \Theta(N/\epsilon)$ sample complexity. We verify both empirically below by sweeping $M$ over three decades and, for the channel, averaging $V_\lambda$ over independent runs.

In [ ]:
def ideal_oracle(f_table):
    """Diagonal Boolean phase oracle O|x> = (-1)^{f(x)} |x>."""
    return np.diag((-1.0) ** f_table.astype(float))


def sketched_oracle(f_table, x_samples):
    """Numpy reference for V = prod_t exp(i tau f(x_t) |x_t><x_t| / M), tau = pi N.

    Each V_t is diagonal, so the product collapses to a single diagonal:
    V_xx = exp(i pi N m_x f(x)) where m_x is the empirical frequency of x.
    """
    n = f_table.size
    m_total = x_samples.size
    counts = np.bincount(x_samples, minlength=n)
    m = counts / m_total
    return np.diag(np.exp(1j * np.pi * n * m * f_table))


# Single realization at M = 4000
M = 4000
x_samples = np.random.randint(0, N, size=M)
V = sketched_oracle(F_TABLE, x_samples)
O = ideal_oracle(F_TABLE)
err = np.linalg.norm(V - O, ord=2)
print(f"||V - O||_2  at  M={M}, N={N}:  {err:.3e}")

# Sweep M to verify the single-realization vs channel-bias scalings.
M_values = np.unique(np.logspace(2, 4.5, 12, dtype=int))

# n_trials = number of independent random sample sequences λ drawn per M.
#   - single_err averages the spectral-norm error over the n_trials draws
#     (just smooths the curve).
#   - channel_err averages the V_λ matrices first and then takes the norm,
#     estimating the channel bias ‖E[V_λ] − O‖. This bias is quadratically
#     smaller than a single realization's fluctuation, so many independent
#     draws are needed for the ± noise to cancel and expose the underlying
#     N/M trend. 200 trials keep the Monte-Carlo residual on channel_err
#     well below the bias across our M range.
n_trials = 200

single_err = np.empty(M_values.size)
channel_err = np.empty(M_values.size)
for k, M_k in enumerate(M_values):
    Vs = [
        sketched_oracle(F_TABLE, np.random.randint(0, N, size=int(M_k)))
        for _ in range(n_trials)
    ]
    single_err[k] = np.mean([np.linalg.norm(Vk - O, ord=2) for Vk in Vs])
    channel_err[k] = np.linalg.norm(np.mean(Vs, axis=0) - O, ord=2)

# Theoretical guides: single-realization ~ sqrt(N/M); channel bias ~ N/M.
c_single = float(np.median(single_err / np.sqrt(N / M_values)))
c_channel = float(np.median(channel_err * M_values / N))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(M_values, single_err, "o-", label=r"single realization $\|V_\lambda - O\|_2$")
ax.loglog(
    M_values, channel_err, "s-", label=r"channel bias $\|\mathbb{E}[V_\lambda] - O\|_2$"
)
ax.loglog(
    M_values,
    c_single * np.sqrt(N / M_values),
    "--",
    color="C0",
    alpha=0.5,
    label=rf"${c_single:.2f}\,\sqrt{{N/M}}$",
)
ax.loglog(
    M_values,
    c_channel * N / M_values,
    "--",
    color="C1",
    alpha=0.5,
    label=rf"${c_channel:.2f}\,N/M$",
)
ax.set_xlabel("Number of samples $M$")
ax.set_ylabel("Spectral-norm error")
ax.set_title(f"Boolean oracle sketching, $N = {N}$, {n_trials} trials")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Extensions

### Non-unirform known probability distribution

The construction above assumed the samples $x_t$ are drawn *uniformly* from $[N]$, which let us pick the global scaling $\tau = \pi N$ so that $\tau m_x \to \pi$. When the samples are drawn from a known, possibly non-uniform distribution $p$ on $[N]$ with $p(x) > 0$ everywhere, the same product-and-concentration argument carries through provided we rescale per sample:

$$
V_t \;=\; \exp\!\Bigl(i\,\tfrac{\pi}{p(x_t)}\,f(x_t)\,|x_t\rangle\!\langle x_t|\Big/M\Bigr),
$$

so that the empirical-frequency limit $m_x \to p(x)$ still gives $\pi$ on each active basis state. 

The sample complexity becomes $M = \Theta\!\bigl(N\,\|1/p\|_\infty / \epsilon\bigr)$, the worst-case rescaling appears as an effective $\|1/p\|_\infty$ factor (maximum value that $1/p(x)$ for $x\in[N]$) that reduces to $N$ for the uniform case.

Bellow we present a generalization of ``quantum_oracle_sketch_boolean`` by accepting a *per-sample*
angle ``thetas[t] = π / (p(x_samples[t]) · M)``; the existing
``apply_basis_phase`` primitive then realises each ``V_t``. ``x_samples``    is assumed pre-filtered to active samples (``f(x_t) = 1``). The function reduces to the uniform Boolean qfunc when ``p ≡ 1/N`` (all angles equal ``π N / M``).

In [ ]:
@qfunc
def quantum_oracle_sketch_known_p(
    thetas: CArray[CReal],
    x_samples: CArray[CInt],
    qvar: QNum,
) -> None:
    """Sketched Boolean phase oracle for a known non-uniform distribution ``p``."""
    repeat(
        count=x_samples.len,
        iteration=lambda t: apply_basis_phase(thetas[t], x_samples[t], qvar),
    )

### Extension to an unknown probability distribution (via QSVT)

When $p$ is not known in advance the per-sample angle $\pi / p(x_t)$ is unavailable. Suppose, however, that we have access to *bounds* on the support of $p$, $p_{\min} \le p(x) \le p_{\max}$ on the support, with **condition number**

$$
\kappa \;:=\; \frac{p_{\max}}{p_{\min}}.
$$

Picking the global scaling $\tau = \pi / p_{\max}$ then places every active angle inside a single half-circle bounded away from $0$:

$$
\tau\,p(x) \in \bigl[\pi/\kappa,\;\pi\bigr],\qquad
V_{xx} \;\longrightarrow\;
\begin{cases}
1, & f(x) = 0, \\[2pt]
e^{i\theta(x)},\ \theta(x) \in \bigl[\pi/\kappa,\;\pi\bigr], & f(x) = 1.
\end{cases}
$$

So the $f = 0$ entries are already correct ($= 1$); the $f = 1$ entries land on a single arc strictly bounded away from $1$ and from wraparound. Since only two kinds of phases appear, we don't need to know $p$ to correct it, we post-process $V$ with a [QSVT](TODO add notebook link) [[4]](#ref4) polynomial that

- maps the eigenvalue $1$ back to $1$ (leave the $f = 0$ entries alone), and
- maps every phase $e^{i\theta(x)}$ with $\theta(x) \in [\pi/\kappa, \pi]$ to $-1$ (force the $f = 1$ entries to the right sign).

This is a *sign-flip on the non-trivial spectrum*: a step / sign-function polynomial applied via QSVT to the block-encoded $V$. The degree of such a polynomial scales as

$$
\deg(\text{poly}) \;=\; \widetilde{\mathcal{O}}(\kappa) \;=\; \widetilde{\mathcal{O}}(p_{\max}/p_{\min}),
$$

i.e. the cost is the **condition number** of $p$ restricted to its support, not just $1/p_{\min}$. Crucially, no classical knowledge of the function $p$ itself enters; only the bounds $p_{\min}, p_{\max}$ are used, and only to fix $\tau$ and the polynomial degree.

When only a *lower* bound on $p_{\min}$ is available (no useful bound on $p_{\max}$), the worst case $p_{\max} \to 1$ gives $\kappa = 1/p_{\min} = \|1/p\|_\infty$, recovering the slack $\widetilde{\mathcal{O}}(\|1/p\|_\infty)$ bound that earlier presentations of the algorithm state.

In [ ]:
# TODO: Implement the QSVT with classiq, mirroring the other work on QSVT,
#  Chebyshev / Remez approximant of $\mathrm{sign}$ on $[\pi/\kappa, \pi]$ and the corresponding Classiq QSVT construction.

## Summary

Quantum oracle sketching converts a stream of classical samples $(x_t, f(x_t))$ into an approximate coherent oracle $V \approx O_f$ in $\Theta(N/\epsilon)$ samples and constant per-sample gate depth, without ever materialising the dataset. The Boolean / uniform case in this notebook is the cleanest setting; the two extensions above (known $p$, unknown $p$) can be twicked to yield a state-sketching, approximating a state peperation operation, and a block-endcoding of a matrix to be used by any downstream quantum query algorithm. Finally, the resulting quantum state can be compressed back to a classical predictor via the classical-shadow tomography, [[2]](#ref2), (or interferometic classical shadow readout [[1]](#ref1)), thereby, completing the three-stage streaming-learning pipeline of [[1]](#ref1).

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. Exponential quantum advantage in processing massive classical data. [arXiv:2604.07639 (2026)](https://arxiv.org/abs/2604.07639)

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). https://doi.org/10.1038/s41567-020-0932-7. [arXiv](https://arxiv.org/abs/2002.08953)

<a id='ref3'></a>
[[3]](#ref3) Giovannetti, V., Lloyd, S., and Maccone, L. *Quantum random access memory.* Physical Review Letters 100, 160501 (2008). https://doi.org/10.1103/PhysRevLett.100.160501. [arXiv](https://arxiv.org/abs/0708.1879)

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366. [arXiv](https://arxiv.org/abs/1806.01838).

<a id='ref5'></a>
[[5]](#ref5) Harrow, A. W., Hassidim, A., and Lloyd, S. *Quantum algorithm for linear systems of equations.* Physical Review Letters 103, 150502 (2009). https://doi.org/10.1103/PhysRevLett.103.150502. [arXiv](https://arxiv.org/abs/0811.3171).

VERIFY THIS IS CORRECT UNDERSTAND THE STATE PREPATION ORACLE BETTER...
### Extension to a Real-Valued Functions for State and Block-Encoding of Matrices

Replacing the Boolean target with a real-valued $\vec v \in \mathbb{R}^N$ generalises the per-sample phase rotation to a continuous angle and, combined with a Hadamard-test ancilla plus oblivious amplitude amplification, yields a *state-preparation* routine $U_v|0\rangle = |\vec v\rangle$. The same primitive extends further to sparse matrices via a pair of element- and index-oracles, giving a *block encoding* of $A$. Both constructions, with full numpy references and Classiq circuits, are covered in the companion notebook [State and Matrix Oracle Sketching](state_and_matrix_oracle_sketching.ipynb).